In [102]:
# Importação de bibliotecas
import pandas as pd
import pickle
import joblib
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

In [103]:
# Modelo ML Regressão Logística
with open('../dados/meu_modelo.pkl', 'rb') as f:
    modelo = pickle.load(f)

In [104]:
# Leitura da nova dataframe para ser tratada e prevista
nova_dataframe_musicas = pd.read_csv('../dados/dataframe_musicas_teste.csv')

In [105]:
# Tratamentos e edições da nova dataframe
# Dropar Duplicatas
nova_dataframe_musicas.drop_duplicates(inplace=True)
# Transformar MS em MIN
nova_dataframe_musicas['duration_ms'] = nova_dataframe_musicas['duration_ms'] / 60000
# Renomear a coluna duration_ms para duration_min
nova_dataframe_musicas.rename(columns={'duration_ms': 'duration_min'}, inplace=True)
# Transformar os gêneros em listas
nova_dataframe_musicas['genres'] = nova_dataframe_musicas['genres'].apply(eval)
# Criar a coluna de Quantidade de Gêneros
nova_dataframe_musicas['qt_genres'] = nova_dataframe_musicas['genres'].apply(len)

In [106]:
# Criação dos dummies da nova dataframe
mlb = MultiLabelBinarizer()
genres_dummies = pd.DataFrame(mlb.fit_transform(nova_dataframe_musicas['genres']), columns=mlb.classes_)

In [107]:
# Dropar colunas irrelevantes
nova_dataframe_musicas = nova_dataframe_musicas.drop(columns=['music', 'genres', 'artist', 'liked'])

In [108]:
# Divisão para futuro escalonamento
colunas_nao_numericas = list(nova_dataframe_musicas.select_dtypes(exclude=['int64', 'float64']).columns)
irrelevantes = colunas_nao_numericas + ['duration_min', 'qt_genres']
numericas = nova_dataframe_musicas.drop(columns=irrelevantes)

In [109]:
# Escalonamento
scaler = joblib.load('../dados/scaler.pkl')
numericas_scaled = scaler.transform(numericas)
numericas_dataframe = pd.DataFrame(numericas_scaled, columns=numericas.columns)

In [110]:
# Concatenação
dados_scaled = pd.concat([nova_dataframe_musicas[irrelevantes].reset_index(drop=True), numericas_dataframe.reset_index(drop=True)], axis=1)

In [111]:
# Concatenação Final
dados_final = pd.concat([dados_scaled, genres_dummies], axis=1)
dados_final

,duration_min,qt_genres,music_popularity,artist_popularity,followers,alternative metal,alternative rock,anime,bedroom pop,blues rock,...,modern blues,mpb,nu disco,nu metal,post-grunge,post-hardcore,rap metal,rap rock,rock,taiwanese indie
0,4.165067,3,0.368452,-0.510044,-0.430155,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
1,3.844433,2,0.486357,-0.609805,-0.522101,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
2,3.236067,2,0.515833,0.088520,-0.161468,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
3,3.274433,3,0.722166,-0.809326,-0.599730,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
4,4.038683,2,0.073690,-2.106216,-0.620103,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
72,3.788217,3,-0.132643,-1.806934,-0.605579,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,1
73,3.631100,5,-0.073690,-0.510044,-0.449603,1,0,0,0,0,...,0,0,0,1,0,0,1,0,0,0
74,3.391767,2,0.899023,-0.510044,-0.497592,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
75,3.028050,2,-0.221071,-1.208369,-0.588053,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0


In [112]:
# Pega todas as colunas do modelo de treinamento
colunas_modelo = modelo.feature_names_in_
# Pega todas as colunas da nova dataframe
colunas_novas = set(dados_final.columns)
# Se a coluna não existir na nova dataframe, adicione com valor padrão 0
for col in colunas_modelo:
    if col not in colunas_novas:
        dados_final[col] = 0
# Identificar colunas extras na nova dataframe
colunas_extra = [col for col in dados_final.columns if col not in colunas_modelo]
# Remover essas colunas extras
dados_final = dados_final.drop(columns=colunas_extra)
# Colocar na mesma ordem
dados_final = dados_final.reindex(columns=colunas_modelo)
dados_final

,duration_min,qt_genres,music_popularity,artist_popularity,followers,acid jazz,acid rock,acoustic pop,alternative metal,alternative rock,...,soul,soul blues,southern gothic,southern rock,stoner rock,surf rock,symphonic metal,synthpop,traditional country,yacht rock
0,4.165067,3,0.368452,-0.510044,-0.430155,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,3.844433,2,0.486357,-0.609805,-0.522101,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,3.236067,2,0.515833,0.088520,-0.161468,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,3.274433,3,0.722166,-0.809326,-0.599730,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,4.038683,2,0.073690,-2.106216,-0.620103,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
72,3.788217,3,-0.132643,-1.806934,-0.605579,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
73,3.631100,5,-0.073690,-0.510044,-0.449603,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
74,3.391767,2,0.899023,-0.510044,-0.497592,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
75,3.028050,2,-0.221071,-1.208369,-0.588053,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [113]:
# Previsão
previsoes = modelo.predict(dados_final)

In [114]:
# Resultados
previsoes

array([0, 1, 1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1])

In [115]:
# Leitura da nova dataframe para ser tratada e prevista
nova_dataframe_musicas = pd.read_csv('../dados/dataframe_musicas_teste.csv')

In [116]:
accuracy = accuracy_score(nova_dataframe_musicas['liked'].values, previsoes)
accuracy

0.5974025974025974

In [117]:
nova_dataframe_musicas['previsão'] = previsoes
nova_dataframe_musicas.to_csv('../dados/resultado.csv', index = False)